In [1]:
# Step 1: Setup MLflow Tracking and Imports
import mlflow
import polars as pl
import plotly.express as px

mlflow.set_tracking_uri("http://localhost:5000")

In [2]:
experiment = mlflow.get_experiment_by_name("contopo")
assert experiment is not None, "Experiment 'contopo' not found!"

In [3]:
models_pd = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id], filter_string="tags.kind = 'metalearner'"
)
metalearners = pl.from_pandas(models_pd)

models_pd = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id], filter_string="tags.kind = 'ensemble'"
)
ensembles = pl.from_pandas(models_pd)

In [4]:
ensembles = ensembles.filter(pl.col("params.method") == "soft")

In [5]:
merged = ensembles.join(
    metalearners, on="tags.component_set_hash", how="inner", suffix="_meta"
)

In [6]:
ens_cons = merged[[
    "tags.rho",
    "metrics.holdout_acc",
    "tags.feature_type_meta",
    "tags.similarity_metric",
    "tags.behaviour_meta",
]]
ens_cons = ens_cons.with_columns(pl.col("tags.rho").cast(pl.Float64))
ens_cons = ens_cons.sort("tags.rho")

In [7]:
ens_cons_filtered = ens_cons.filter(
    (pl.col("tags.feature_type_meta") == "embeddings+profiles")
    & (pl.col("tags.similarity_metric") == "cosine")
)
ens_cons_filtered = ens_cons_filtered.sort("tags.rho")

In [8]:
ensemble_acc = ensembles.select(["tags.rho", "metrics.ensemble_accuracy"])
ensemble_acc = ensemble_acc.with_columns(pl.col("tags.rho").cast(pl.Float64))

ens_cons_filtered = ens_cons_filtered.join(ensemble_acc, on="tags.rho", how="left")
ens_cons_filtered = ens_cons_filtered.sort("tags.rho")

In [9]:
plot_df = ens_cons_filtered.melt(
    id_vars=["tags.rho", "tags.behaviour_meta"],
    value_vars=["metrics.holdout_acc", "metrics.ensemble_accuracy"],
    variable_name="metric_type",
    value_name="accuracy",
)

plot_df = plot_df.with_columns(
    pl.when(pl.col("metric_type") == "metrics.ensemble_accuracy")
    .then(pl.lit("Ensemble Accuracy"))
    .otherwise(pl.col("tags.behaviour_meta"))
    .alias("line_type")
)

/tmp/ipykernel_74393/2069513009.py:1: DeprecationWarning: `DataFrame.melt` is deprecated; use `DataFrame.unpivot` instead, with `index` instead of `id_vars` and `on` instead of `value_vars`
  plot_df = ens_cons_filtered.melt(


In [10]:
# Convert 'tags.rho' to str for categorical axis
plot_df = plot_df.with_columns(
    pl.col("tags.rho").cast(pl.Utf8)
)

fig = px.line(
    plot_df,
    x="tags.rho",  # Now categorical
    y="accuracy",
    color="line_type",
    markers=True,
    title="Meta Inference and Ensemble Accuracy by Rho",
    labels={
        "tags.rho": "Rho",
        "accuracy": "Accuracy",
        "line_type": "Type",
    },
)
fig.update_traces(marker=dict(size=10, opacity=0.8))
fig.update_layout(template="simple_white", xaxis_type="category")
fig.show()